# Étape 1 — Exploration des données images (Personne 1 : Vision)
Projet : détection des maladies des plantes + assistant textuel (RAG)

Datasets : **PlantVillage** (images de laboratoire, ~54k, 38 classes) et **PlantDoc** (images de terrain, ~2,6k, 27 classes).

Contenu : 1) chargement, 2) distribution des classes, 3) grille d'exemples, 4) dimensions et formats, 5) images corrompues et doublons, 6) correspondance des classes PlantVillage / PlantDoc, 7) synthèse.

In [ ]:
# !pip install pandas matplotlib pillow imagehash tqdm
import os, hashlib, random
from pathlib import Path
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

random.seed(42)

# A ADAPTER : dossiers où vous avez téléchargé les datasets (un sous-dossier par classe)
PV_DIR = Path("data/PlantVillage")        # ex. Kaggle : "PlantVillage dataset" (color)
PD_TRAIN = Path("data/PlantDoc/train")    # ex. PlantDoc-Dataset (train/)
PD_TEST = Path("data/PlantDoc/test")
EXTS = {".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"}

## 1. Chargement : une ligne par image

In [ ]:
def index_dataset(root, name):
    rows = []
    for cls_dir in sorted(p for p in Path(root).iterdir() if p.is_dir()):
        for f in cls_dir.iterdir():
            if f.suffix in EXTS:
                rows.append({"dataset": name, "classe": cls_dir.name, "chemin": str(f)})
    return pd.DataFrame(rows)

df_pv = index_dataset(PV_DIR, "PlantVillage")
df_pd = pd.concat([index_dataset(PD_TRAIN, "PlantDoc-train"),
                   index_dataset(PD_TEST, "PlantDoc-test")], ignore_index=True)
df = pd.concat([df_pv, df_pd], ignore_index=True)

print(df.groupby("dataset").agg(images=("chemin", "count"), classes=("classe", "nunique")))
df.head()

## 2. Distribution des classes (visualisation 1)

In [ ]:
def plot_class_distribution(d, title):
    counts = d["classe"].value_counts().sort_values()
    fig, ax = plt.subplots(figsize=(9, max(4, 0.28 * len(counts))))
    counts.plot.barh(ax=ax)
    ax.set_title(title); ax.set_xlabel("Nombre d'images")
    plt.tight_layout(); plt.show()
    print(f"Min : {counts.min()} | Max : {counts.max()} | Ratio déséquilibre max/min : {counts.max()/counts.min():.1f}")

plot_class_distribution(df_pv, "PlantVillage : images par classe")
plot_class_distribution(df_pd[df_pd.dataset == "PlantDoc-train"], "PlantDoc (train) : images par classe")

## 3. Grille d'exemples : laboratoire vs terrain (visualisation 2)

In [ ]:
def sample_grid(d, n=8, title=""):
    sample = d.sample(n, random_state=1)
    cols = 4; rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(3.2 * cols, 3.2 * rows))
    for ax, (_, r) in zip(axes.ravel(), sample.iterrows()):
        ax.imshow(Image.open(r["chemin"]).convert("RGB"))
        ax.set_title(r["classe"][:28], fontsize=8); ax.axis("off")
    for ax in axes.ravel()[n:]: ax.axis("off")
    fig.suptitle(title); plt.tight_layout(); plt.show()

sample_grid(df_pv, 8, "PlantVillage (fond uniforme, laboratoire)")
sample_grid(df_pd, 8, "PlantDoc (conditions réelles)")

## 4. Dimensions, formats et modes couleur

In [ ]:
def image_stats(d, max_n=2000):
    d = d.sample(min(max_n, len(d)), random_state=0)
    out = []
    for p in tqdm(d["chemin"]):
        try:
            with Image.open(p) as im:
                out.append({"chemin": p, "largeur": im.width, "hauteur": im.height,
                            "mode": im.mode, "format": im.format})
        except Exception as e:
            out.append({"chemin": p, "erreur": str(e)})
    return pd.DataFrame(out)

stats_pv = image_stats(df_pv); stats_pd = image_stats(df_pd)
for name, s in [("PlantVillage", stats_pv), ("PlantDoc", stats_pd)]:
    print(name); print(s[["largeur", "hauteur"]].describe().round(0))
    print(s["mode"].value_counts().to_dict(), s["format"].value_counts().to_dict(), "\n")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(stats_pv.largeur, stats_pv.hauteur, s=4, alpha=.4); axes[0].set_title("PlantVillage")
axes[1].scatter(stats_pd.largeur, stats_pd.hauteur, s=4, alpha=.4); axes[1].set_title("PlantDoc")
for a in axes: a.set_xlabel("Largeur (px)"); a.set_ylabel("Hauteur (px)")
plt.tight_layout(); plt.show()

## 5. Images corrompues et doublons
- **Doublons exacts** : hash MD5 du fichier.
- **Quasi-doublons** : hash perceptuel (`imagehash`). Dans PlantVillage, la même feuille est souvent photographiée plusieurs fois, d'où un risque de **fuite de données** (data leakage) si le découpage train/test est aléatoire.

In [ ]:
def md5(p):
    h = hashlib.md5()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""): h.update(chunk)
    return h.hexdigest()

df["md5"] = [md5(p) for p in tqdm(df["chemin"])]
dups = df[df.duplicated("md5", keep=False)].sort_values("md5")
print("Doublons exacts :", len(dups), "fichiers dans", dups["md5"].nunique(), "groupes")
print(dups.groupby("dataset").size())

# Doublons entre PlantDoc train et test (fuite possible)
tr = set(df[df.dataset == "PlantDoc-train"].md5); te = set(df[df.dataset == "PlantDoc-test"].md5)
print("Images communes PlantDoc train/test :", len(tr & te))

In [ ]:
import imagehash
def phash(p):
    try:
        with Image.open(p) as im: return str(imagehash.phash(im.convert("RGB")))
    except Exception: return None

sub = df_pv.sample(min(5000, len(df_pv)), random_state=0).copy()
sub["phash"] = [phash(p) for p in tqdm(sub["chemin"])]
near = sub[sub.duplicated("phash", keep=False) & sub.phash.notna()]
print("Quasi-doublons (phash identique) dans l'échantillon PlantVillage :", len(near))

## 6. Correspondance des classes PlantVillage / PlantDoc (base du test de généralisation)

In [ ]:
import re
def norm(s):
    s = s.lower().replace("___", " ").replace("_", " ")
    return re.sub(r"[^a-z ]", " ", s).split()

pv_classes = sorted(df_pv.classe.unique()); pd_classes = sorted(df_pd.classe.unique())
print("PlantVillage :", pv_classes[:5], "...")
print("PlantDoc     :", pd_classes[:5], "...")

# Mapping automatique approximatif : plus grand recouvrement de mots. A VERIFIER MANUELLEMENT.
mapping = {}
for c in pd_classes:
    wc = set(norm(c))
    best = max(pv_classes, key=lambda p: len(wc & set(norm(p))))
    score = len(wc & set(norm(best)))
    mapping[c] = best if score >= 2 else None
pd.Series(mapping, name="classe PlantVillage proche").to_frame()